In [1]:
from pathlib import Path

from pycirclize import Circos
from scipy.io import loadmat
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
def find_behav_cluster(behav_name, behav_cluster_dict):
    behav_fname = behav_name.replace(" ", "_")
    behav_fname = behav_fname.replace("/", "-")
    cluster_curr = "class 0"
    for cluster, behav_list in behav_cluster_dict.items():
        if behav_fname in behav_list:
            cluster_curr = cluster
    return cluster_curr

def find_pos(behav_name, cluster, behav_cluster_dict):
    for i, behav in enumerate(behav_cluster_dict[cluster]):
        if behav == behav_name:
            pos = i + 0.5
    return pos

def read_corr(dataset, method, cbpp_folder):
    behav_names = pd.read_table(Path("lists", f"{dataset}_colloquial_list.txt"), delimiter="\t", header=None).squeeze()
    behav_cluster = pd.read_csv(Path("results", dataset, "lists", f"behavior_cls_{method}_above0.3.csv"))
    behav_cluster = {col: list(behav_cluster[col].dropna()) for col in behav_cluster.columns} | {"class 0": []}
    err_corr = loadmat(Path("results", dataset, "cbpp", cbpp_folder, f"corr_PredErr_{method}.mat"))["err_corr"]
    
    behav_cluster_plot = {cluster: [] for cluster in behav_cluster.keys()}
    for behav in behav_names:
        cluster = find_behav_cluster(behav, behav_cluster)
        behav_cluster_plot[cluster].append(behav)
    
    n_behav = len(behav_names)
    behav_corr = []
    pair_ind = 0
    for i in range(n_behav-1):
        for j in range(i+1, n_behav):
            corr_curr = {
                "Behavior 1": behav_names[i], "Behavior 2": behav_names[j], "Correlation of prediction errors": err_corr[0, pair_ind],
                "Behavior 1 cluster": find_behav_cluster(behav_names[i], behav_cluster),
                "Behavior 2 cluster": find_behav_cluster(behav_names[j], behav_cluster)}
            behav_corr.append(pd.DataFrame(corr_curr, index=[pair_ind]))
            pair_ind += 1
    behav_corr = pd.concat(behav_corr, axis="index")

    return behav_corr, behav_cluster_plot

In [3]:
cbpp_folders = {"ABCD": "4278sub_34behaviors", "HCP-D": "432sub_22behaviors", "HCP": "784sub_51behaviors"}
# Color code by cohort so that clusters of similar domains have similar colors
sector_color = {
    "ABCD": {"class 0": "lightgrey", "class 1": "mediumseagreen", "class 2": "limegreen", "class 3": "orange", "class 4": "pink"},
    "HCP-D": {"class 0": "lightgrey", "class 1": "limegreen", "class 2": "skyblue"},
    "HCP": {
        "class 0": "lightgrey", "class 1": "mediumseagreen", "class 2": "darkturquoise", "class 3": "skyblue", "class 4": "deepskyblue"}}

for dataset in ["ABCD", "HCP-D", "HCP"]:
    for method in ["SVR", "KRR"]:
        behav_corr, behav_cluster_plot = read_corr(dataset, method, cbpp_folders[dataset])
        sectors = {cluster: len(behav_list) for cluster, behav_list in behav_cluster_plot.items()}
        circos = Circos(sectors)
        for sector in circos.sectors:
            track = sector.add_track((95, 100))
            track.axis(fc=sector_color[dataset][sector.name], ec="white")
        
            tick_pos = list(x+0.5 for x in range(len(behav_cluster_plot[sector.name])))
            track.xticks(
                tick_pos, behav_cluster_plot[sector.name], label_orientation="vertical", label_size=12, text_kws={"wrap": True})
        
        thresh = 0.3
        for _, corr_row in behav_corr.iterrows():
            width = abs(corr_row["Correlation of prediction errors"]) * 5
            if corr_row["Correlation of prediction errors"] > thresh:
                color = sector_color[dataset][corr_row["Behavior 1 cluster"]]
                alpha = 1
            else:
                color = "lightgrey"
                alpha = 0.5
            
            pos_1 = find_pos(corr_row["Behavior 1"], corr_row["Behavior 1 cluster"], behav_cluster_plot)
            pos_2 = find_pos(corr_row["Behavior 2"], corr_row["Behavior 2 cluster"], behav_cluster_plot)
            
            circos.link_line(
                (corr_row["Behavior 1 cluster"], pos_1), (corr_row["Behavior 2 cluster"], pos_2),
                color=color, alpha=alpha, lw=width, height_ratio=0.1)
        
        f = circos.plotfig()
        plt.savefig(Path("plots", f"{dataset}_{method}_cluster.png"), dpi=500)
        plt.savefig(Path("plots", f"{dataset}_{method}_cluster.svg"), format="svg")
        plt.close()